In [1]:
# Clone repos + install
!git clone https://github.com/Anand-786/llm-quantization-thesis.git
%cd /content/llm-quantization-thesis
!git clone https://github.com/mit-han-lab/smoothquant.git smoothquant_repo
!pip uninstall smoothquant -y
!cd smoothquant_repo && pip install -e .
!pip install -q transformers accelerate datasets zstandard tqdm

# Mount Drive
from google.colab import drive
drive.mount('/content/drive')

# Pile validation set (calibration data)
!mkdir -p smoothquant_repo/dataset
!wget -q -O smoothquant_repo/dataset/val.jsonl.zst \
    https://huggingface.co/datasets/mit-han-lab/pile-val-backup/resolve/main/val.jsonl.zst

# Drive output dir for the per-p percentile files
!mkdir -p /content/drive/MyDrive/thesis_results/act_percentiles/opt-2.7b

# Verify
!nvidia-smi
!ls -la /content/drive/MyDrive/thesis_results/act_scales/opt-2.7b.pt
!ls -la smoothquant_repo/dataset/val.jsonl.zst

Cloning into 'llm-quantization-thesis'...
remote: Enumerating objects: 97, done.
remote: Counting objects: 100% (97/97), done.
remote: Compressing objects: 100% (67/67), done.
remote: Total 97 (delta 29), reused 87 (delta 19), pack-reused 0 (from 0)
Receiving objects: 100% (97/97), 4.81 MiB | 22.18 MiB/s, done.
Resolving deltas: 100% (29/29), done.
/content/llm-quantization-thesis
Cloning into 'smoothquant_repo'...
remote: Enumerating objects: 352, done.
remote: Counting objects: 100% (169/169), done.
remote: Compressing objects: 100% (79/79), done.
remote: Total 352 (delta 120), reused 90 (delta 90), pack-reused 183 (from 1)
Receiving objects: 100% (352/352), 6.80 MiB | 17.36 MiB/s, done.
Resolving deltas: 100% (202/202), done.
Obtaining file:///content/llm-quantization-thesis/smoothquant_repo
  Preparing metadata (setup.py) ... done
  Running setup.py develop for smoothquant
Mounted at /content/drive
Sun May  3 17:01:49 2026       
+---------------------------------------------------

In [2]:
%%writefile /content/percentile_calibration.py
"""Per-channel exact-percentile calibration via top-K buffers (Task 02)."""
import functools
import math
from typing import Dict, Iterable, List

import torch
import torch.nn as nn
from datasets import load_dataset
from tqdm import tqdm


def _smoothing_site_names(model):
    from transformers.models.opt.modeling_opt import OPTDecoderLayer
    sites = []
    for name, module in model.named_modules():
        if isinstance(module, OPTDecoderLayer):
            sites.append(f"{name}.self_attn.q_proj")
            sites.append(f"{name}.fc1")
    return sites


@torch.no_grad()
def get_act_percentiles(
    model,
    tokenizer,
    dataset_path,
    percentiles=(1.0, 0.999, 0.995, 0.99, 0.95, 0.90),
    num_samples=512,
    seq_len=512,
    buffer_device="cuda",
    buffer_dtype=torch.float16,
    safety_margin=16,
):
    p_list = sorted(set(float(p) for p in percentiles))
    if not p_list:
        raise ValueError("`percentiles` must be non-empty")
    for p in p_list:
        if not (0.0 < p <= 1.0):
            raise ValueError(f"percentile {p} must be in (0, 1]")

    p_min = p_list[0]
    n_total = num_samples * seq_len
    k = max(2, min(n_total, math.ceil((1.0 - p_min) * n_total) + safety_margin))

    model.eval()
    device = next(model.parameters()).device

    site_names = _smoothing_site_names(model)
    if not site_names:
        raise RuntimeError("No OPT decoder layers found — is this an OPT model?")
    sites = {name: None for name in site_names}
    for name, module in model.named_modules():
        if name in sites and isinstance(module, nn.Linear):
            sites[name] = module
    missing = [n for n, m in sites.items() if m is None]
    if missing:
        raise RuntimeError(f"Unresolved smoothing sites: {missing[:3]}...")

    buffers = {}
    for name, lin in sites.items():
        buffers[name] = torch.full(
            (k, lin.in_features),
            fill_value=float("-inf"),
            dtype=buffer_dtype,
            device=buffer_device,
        )

    def update_buffer(name, x):
        in_features = x.shape[-1]
        new = x.reshape(-1, in_features).abs().to(buffer_dtype)
        if buffer_device == "cpu":
            new = new.cpu()
        elif new.device.type != "cuda":
            new = new.to(buffer_device)
        combined = torch.cat([buffers[name], new], dim=0)
        topk = torch.topk(combined, k=k, dim=0, largest=True, sorted=False).values
        buffers[name].copy_(topk)

    def stat_input_hook(_m, inputs, _output, name):
        x = inputs[0] if isinstance(inputs, tuple) else inputs
        if isinstance(x, tuple):
            x = x[0]
        update_buffer(name, x)

    hooks = []
    for name, lin in sites.items():
        hooks.append(lin.register_forward_hook(functools.partial(stat_input_hook, name=name)))

    try:
        dataset = load_dataset("json", data_files=dataset_path, split="train")
        dataset = dataset.shuffle(seed=42)
        for i in tqdm(range(num_samples), desc="Top-K calibration"):
            input_ids = tokenizer(
                dataset[i]["text"],
                return_tensors="pt",
                max_length=seq_len,
                truncation=True,
            ).input_ids.to(device)
            model(input_ids)
    finally:
        for h in hooks:
            h.remove()

    out = {f"{p:g}": {} for p in p_list}
    for name, buf in buffers.items():
        sorted_buf = torch.sort(buf, dim=0).values
        k_eff = sorted_buf.shape[0]
        for p in p_list:
            pos = p * (n_total - 1)
            idx_f = float(k_eff) - float(n_total) + pos
            if idx_f < 0:
                raise ValueError(
                    f"p={p} below buffer floor (k={k_eff}, N={n_total})"
                )
            lo = max(0, min(k_eff - 1, math.floor(idx_f)))
            hi = max(0, min(k_eff - 1, math.ceil(idx_f)))
            if hi == lo:
                val = sorted_buf[lo].clone()
            else:
                w = idx_f - lo
                val = sorted_buf[lo].float() * (1.0 - w) + sorted_buf[hi].float() * w
                val = val.to(sorted_buf.dtype)
            out[f"{p:g}"][name] = val.detach().cpu()
    return out

Writing /content/percentile_calibration.py


In [3]:
import sys, os, time
sys.path.insert(0, "/content")

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from percentile_calibration import get_act_percentiles

MODEL = "facebook/opt-2.7b"
ORIG_MAX_PATH = "/content/drive/MyDrive/thesis_results/act_scales/opt-2.7b.pt"
DATASET_PATH = "/content/llm-quantization-thesis/smoothquant_repo/dataset/val.jsonl.zst"
OUT_DIR = "/content/drive/MyDrive/thesis_results/act_percentiles/opt-2.7b"

# Calibrate p=1.0 in-memory for the validation diff, then drop it before saving.
PERCENTILES_TO_SAVE = [0.999, 0.995, 0.99, 0.95, 0.90]
PERCENTILES_FOR_CALIBRATION = [1.0] + PERCENTILES_TO_SAVE

print("Loading tokenizer + model (fp16, GPU)...")
tokenizer = AutoTokenizer.from_pretrained(MODEL)
model = AutoModelForCausalLM.from_pretrained(
    MODEL, torch_dtype=torch.float16, device_map="auto"
)
model.eval()

print(f"Calibrating exact percentiles {PERCENTILES_FOR_CALIBRATION} via top-K buffer ...")
t0 = time.time()
act_pct = get_act_percentiles(
    model=model,
    tokenizer=tokenizer,
    dataset_path=DATASET_PATH,
    percentiles=PERCENTILES_FOR_CALIBRATION,
    num_samples=512,
    seq_len=512,
    buffer_device="cuda",        # OPT-2.7B buffers ~8 GB on A100, comfortable
    buffer_dtype=torch.float16,
)
print(f"Calibration done in {time.time() - t0:.1f}s")

# --- Pipeline-correctness check: in-memory diff of p=1.0 vs upstream max ---
orig = torch.load(ORIG_MAX_PATH)
ours_max = act_pct["1"]
common = [k for k in ours_max.keys() if k in orig]
print(f"\nValidating top-K pipeline against upstream max ({len(common)} sites)...")
max_abs_diff = 0.0
max_rel_diff = 0.0
worst_name = None
for name in common:
    a = orig[name].float().cpu()
    b = ours_max[name].float().cpu()
    abs_diff = (a - b).abs()
    rel_diff = abs_diff / a.clamp(min=1e-8)
    if abs_diff.max().item() > max_abs_diff:
        max_abs_diff = abs_diff.max().item()
        worst_name = name
    max_rel_diff = max(max_rel_diff, rel_diff.max().item())
print(f"  worst abs diff: {max_abs_diff:.6f}  (channel in '{worst_name}')")
print(f"  worst rel diff: {max_rel_diff:.6f}")
print(f"  expected: rel diff ~= one fp16 ULP (~5e-3 worst case). >1e-2 means a real bug.")
assert max_rel_diff < 1e-2, "Top-K p=1.0 disagrees with upstream max — investigate before trusting percentile rows."

# --- Save only p<1.0 files. p=1.0 is fp16-noisy; baseline comes from Task 01. ---
os.makedirs(OUT_DIR, exist_ok=True)
print(f"\nSaving {len(PERCENTILES_TO_SAVE)} per-p files (p=1.0 deliberately skipped):")
for p in PERCENTILES_TO_SAVE:
    p_key = f"{p:g}"
    out_path = os.path.join(OUT_DIR, f"p{p_key}.pt")
    torch.save(act_pct[p_key], out_path)
    size_mb = os.path.getsize(out_path) / (1024 ** 2)
    print(f"  saved p={p_key:>6} -> {out_path}  ({size_mb:.2f} MB)")

print("\nKeys per file (sample):", list(act_pct[f"{PERCENTILES_TO_SAVE[0]:g}"].keys())[:3])

Loading tokenizer + model (fp16, GPU)...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/691 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/685 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

generation_config.json:   0%|          | 0.00/137 [00:00<?, ?B/s]

Calibrating exact percentiles [1.0, 0.999, 0.995, 0.99, 0.95, 0.9] via top-K buffer ...


Generating train split: 0 examples [00:00, ? examples/s]

Top-K calibration: 100%|██████████| 512/512 [11:42<00:00,  1.37s/it]


Calibration done in 729.0s

Validating top-K pipeline against upstream max (64 sites)...
  worst abs diff: 0.000000  (channel in 'None')
  worst rel diff: 0.000000
  expected: rel diff ~= one fp16 ULP (~5e-3 worst case). >1e-2 means a real bug.

Saving 5 per-p files (p=1.0 deliberately skipped):
  saved p= 0.999 -> /content/drive/MyDrive/thesis_results/act_percentiles/opt-2.7b/p0.999.pt  (0.33 MB)
  saved p= 0.995 -> /content/drive/MyDrive/thesis_results/act_percentiles/opt-2.7b/p0.995.pt  (0.33 MB)
  saved p=  0.99 -> /content/drive/MyDrive/thesis_results/act_percentiles/opt-2.7b/p0.99.pt  (0.33 MB)
  saved p=  0.95 -> /content/drive/MyDrive/thesis_results/act_percentiles/opt-2.7b/p0.95.pt  (0.33 MB)
  saved p=   0.9 -> /content/drive/MyDrive/thesis_results/act_percentiles/opt-2.7b/p0.9.pt  (0.33 MB)

Keys per file (sample): ['model.decoder.layers.0.self_attn.q_proj', 'model.decoder.layers.0.fc1', 'model.decoder.layers.1.self_attn.q_proj']


In [4]:
import torch, os

OUT_DIR = "/content/drive/MyDrive/thesis_results/act_percentiles/opt-2.7b"
for fname in sorted(os.listdir(OUT_DIR)):
    if not fname.endswith(".pt"):
        continue
    path = os.path.join(OUT_DIR, fname)
    d = torch.load(path)
    sample_name = next(iter(d.keys()))
    v = d[sample_name]
    print(f"{fname:>14} | {len(d):>3} sites | sample {sample_name} -> shape {tuple(v.shape)}, max={v.max().item():.4f}, mean={v.mean().item():.4f}")

       p0.9.pt |  64 sites | sample model.decoder.layers.0.self_attn.q_proj -> shape (2560,), max=4.2461, mean=1.4355
      p0.95.pt |  64 sites | sample model.decoder.layers.0.self_attn.q_proj -> shape (2560,), max=5.0742, mean=1.7461
      p0.99.pt |  64 sites | sample model.decoder.layers.0.self_attn.q_proj -> shape (2560,), max=6.4414, mean=2.3379
     p0.995.pt |  64 sites | sample model.decoder.layers.0.self_attn.q_proj -> shape (2560,), max=6.8789, mean=2.5645
     p0.999.pt |  64 sites | sample model.decoder.layers.0.self_attn.q_proj -> shape (2560,), max=7.8125, mean=3.0508
